# Ayanami teacher bulk cache — Colab (self-contained)
Writes the full 1,911-prompt pool from the embedded payload, then caches
shard ranges. Run ONE range per session (each ~40 min on T4).
Suggested ranges: 0-4, 5-9, 10-14, 15-19 (shard_size 100).
Keep the tab active; reruns resume finished IDs. OUT must stay on Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
import os
REPO = '/content/Ayanami'
if not os.path.isdir(REPO):
    !git clone https://github.com/Sarahi-M-00/Ayanami.git /content/Ayanami
%cd /content/Ayanami
!git pull --ff-only 2>&1 | tail -1
!bash scripts/setup_remote.sh


In [ ]:
# One-time per VM (~10 min). Re-run after every VM recycle.
!sudo apt-get update -qq && sudo apt-get install -qq -y python3.13-venv
!rm -rf .venv
!bash scripts/setup_remote.sh  # ends with: check_env PASSED


In [ ]:
import json, base64, zlib
POOL = json.loads(zlib.decompress(base64.b64decode(
    '...')).decode())
assert len(POOL['prompts']) == 1911, len(POOL['prompts'])
print('POOL ready:', len(POOL['prompts']), 'prompts')


In [ ]:
import json, os
os.makedirs('data/processed', exist_ok=True)
persona = open('persona/system_prompt.md', encoding='utf-8').read()
sys_msg = {'role': 'system', 'content': persona}
recs = []
for c in POOL['prompts']:
    usr = {'role': 'user', 'content': c['user']}
    teacher = [sys_msg, usr]
    student = [usr] if c['bare'] else [sys_msg, usr]
    recs.append({'id': c['id'], 'category': c['category'], 'lang': c['lang'],
               'messages': teacher, 'student_messages': student,
               'verifier': c['verifier'], 'split': c['split'], 'source': c['source']})
with open('data/processed/prompts_v1.jsonl', 'w', encoding='utf-8') as fh:
    fh.write('\n'.join(json.dumps(r, ensure_ascii=False) for r in recs) + '\n')
print('prompts_v1.jsonl written:', len(recs))


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab Secrets')
except Exception as exc:
    print('set HF_TOKEN first:', exc)
%env HF_HOME=/tmp/hf_cache
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ayanami/teacher_cache/qwen-qwen3-8b/b968826d9c46dd6066d109eabc6255188de91218/'
print('OUT =', OUT)


In [ ]:
.venv/bin/python scripts/teacher_cache.py preflight --prompts data/processed/prompts_v1.jsonl


In [ ]:
# EDIT the range per session: 0-4, 5-9, 10-14, then 15-19.
SHARDS = '0-4'
.venv/bin/python scripts/teacher_cache.py cache --prompts data/processed/prompts_v1.jsonl --out "$OUT" --shards "{SHARDS}" --seed 7


In [ ]:
!ls "$OUT" 2>/dev/null || echo 'check OUT path'
# Copy new shard_*.jsonl + manifest_*.json (+ rejected_*.json) back to the laptop repo path.
# Record GPU hours in docs/GPU_LEDGER.md and tell the agent.
